In [14]:
import pandas as pd

df = pd.read_csv("data/teams.csv")

# Clean team names
df["team"] = (
    df["team"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.replace("*", "", regex=False)
    .str.strip()
)

df["team"] = df["team"].replace({"Albany (NY)": "Albany"})

# Remove accidental repeated headers if they exist
df = df[df["team"] != "School"].copy()

# Coerce numeric columns
num_cols = df.columns.drop("team")
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# Drop separator columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Rename repeated columns to match Sports-Reference structure
df = df.rename(columns={
    "wins": "W",
    "losses": "L",
    "wins.1": "Conf_W",
    "losses.1": "Conf_L",
    "wins.2": "Home_W",
    "losses.2": "Home_L",
    "wins.3": "Away_W",
    "losses.3": "Away_L",
    "Tm.": "Points_For",
    "Opp.": "Points_Against",
})
print(df.columns)
print(df.head())

Index(['team', 'G', 'W', 'L', 'win_pct', 'SRS', 'SOS', 'Conf_W', 'Conf_L',
       'Home_W', 'Home_L', 'Away_W', 'Away_L', 'Points_For', 'Points_Against',
       'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', 'FT', 'FTA', 'FT%', 'ORB',
       'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'year'],
      dtype='object')
       team   G   W   L  win_pct    SRS   SOS  Conf_W  Conf_L  Home_W  ...  \
0    Albany  33  24   9    0.727  -0.34 -5.18      15       1      12  ...   
1   Arizona  38  34   4    0.895  24.33  7.41      16       2      17  ...   
2  Arkansas  36  27   9    0.750  14.07  6.79      13       5      17  ...   
3    Baylor  34  24  10    0.706  17.85  9.33      11       7      16  ...   
4   Belmont  33  22  11    0.667   0.35 -2.77      11       5      12  ...   

   FTA    FT%  ORB   TRB  AST  STL  BLK  TOV   PF  year  
0  650  0.758  334  1111  349  187   53  381  544  2015  
1  974  0.719  411  1399  528  275  131  419  681  2015  
2  819  0.724  470  1290  579  277  168  430

In [26]:
########################################
# 1. LOAD AND CLEAN TEAM SEASON STATS
########################################

df = pd.read_csv("data/teams.csv")

# Clean team names
df["team"] = (
    df["team"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.replace("*", "", regex=False)
    .str.strip()
)

# Normalize known name variants
df["team"] = df["team"].replace({
    "Albany (NY)": "Albany"
})

# Remove accidental header rows
df = df[df["team"] != "School"].copy()

# Coerce numeric columns
num_cols = df.columns.drop("team")
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# Drop separator columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Rename repeated columns
df = df.rename(columns={
    "wins": "W",
    "losses": "L",
    "wins.1": "Conf_W",
    "losses.1": "Conf_L",
    "wins.2": "Home_W",
    "losses.2": "Home_L",
    "wins.3": "Away_W",
    "losses.3": "Away_L",
    "Tm.": "Points_For",
    "Opp.": "Points_Against",
})

########################################
# 2. LOAD TOURNAMENT GAMES
########################################

games = pd.read_csv("data/tournament_games.csv")

# Normalize team names in tournament data
games["winner"] = games["winner"].replace({
    "Albany (NY)": "Albany"
})
games["loser"] = games["loser"].replace({
    "Albany (NY)": "Albany"
})

########################################
# 3. MERGE TEAM STATS (WINNER + LOSER)
########################################

# Merge winner stats
games_w = games.merge(
    df,
    left_on=["winner", "season"],
    right_on=["team", "year"],
    how="left"
)

# Merge loser stats
games_wl = games_w.merge(
    df,
    left_on=["loser", "season"],
    right_on=["team", "year"],
    how="left",
    suffixes=("_winner", "_loser")
)

########################################
# 4. CREATE DIFFERENCE FEATURES
########################################

features = [
    "SRS",
    "win_pct",
    "FG%",
    "3P%",
    "FT%",
    "TRB",
    "AST",
    "TOV"
]

for col in features:
    games_wl[f"{col}_diff"] = (
        games_wl[f"{col}_winner"] -
        games_wl[f"{col}_loser"]
    )

# Seed difference (lower seed = stronger team)
games_wl["seed_diff"] = (
    games_wl["winner_seed"] -
    games_wl["loser_seed"]
)

########################################
# 5. CREATE SYMMETRIC DATASET
########################################

# Winner perspective
games_wl["win_label"] = 1

# Loser perspective
loser_view = games_wl.copy()

for col in features:
    loser_view[f"{col}_diff"] *= -1

loser_view["seed_diff"] *= -1
loser_view["win_label"] = 0

# Combine both perspectives
model_df = pd.concat([games_wl, loser_view], ignore_index=True)

########################################
# 6. BUILD FINAL ML DATASET
########################################

# Select difference features (remove duplicates safely)
feature_cols = sorted(set(
    col for col in model_df.columns
    if col.endswith("_diff")
))

final_df = model_df[feature_cols + ["win_label", "season"]]

# Drop rows with failed merges
final_df = final_df.dropna()

########################################
# 7. FINAL SANITY CHECKS
########################################

print("Final dataset shape:", final_df.shape)
print("Total NaNs:", final_df.isna().sum().sum())
print(final_df.head())

########################################
# 8. SAVE TO CSV
########################################

final_df.to_csv("data/tournament_model_ml.csv", index=False)

Final dataset shape: (1068, 11)
Total NaNs: 0
   3P%_diff  AST_diff  FG%_diff  FT%_diff  SRS_diff  TOV_diff  TRB_diff  \
0    -0.019      56.0    -0.015    -0.036     26.50      16.0     218.0   
2     0.072     -88.0     0.019     0.031      8.73     -29.0      24.0   
3    -0.082     -61.0    -0.032    -0.016     14.49      12.0     153.0   
4     0.044      16.0     0.024    -0.021     -2.73       8.0    -107.0   
5    -0.015      81.0    -0.003    -0.017     18.99      38.0     221.0   

   seed_diff  win_pct_diff  win_label  season  
0        -15         0.311          1    2015  
2         -7         0.172          1    2015  
3         -9         0.132          1    2015  
4          5         0.103          1    2015  
5        -11        -0.041          1    2015  
